# Folio count growth — line chart (Industry)
Folio count from **Jan 2022 (~13.26 Cr)** to **Dec 2025 (~26.12 Cr)** with key milestone markers.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

# --- repo-root detection (works from any notebook working directory) ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(10):
        if (cand / 'Data' / 'processed' / 'industry_folio_count_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'industry_folio_count_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_PATH = _REPO_ROOT / 'Data' / 'processed' / 'industry_folio_count_clean.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing file: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
df['month'] = pd.to_datetime(df['month'], errors='coerce')
df['total_folios_crore'] = pd.to_numeric(df['total_folios_crore'], errors='coerce')
df = df.dropna(subset=['month', 'total_folios_crore']).sort_values('month')

# --- Milestones ---
m_start = pd.Timestamp('2022-01-01')
m_end = pd.Timestamp('2025-12-01')

def _value_at(month_ts: pd.Timestamp) -> float:
    row = df.loc[df['month'] == month_ts]
    if row.empty:
        return float('nan')
    return float(row['total_folios_crore'].iloc[0])

start_val = _value_at(m_start)
end_val = _value_at(m_end)

thresholds = [15.0, 20.0, 25.0]
milestones = []

for t in thresholds:
    hit = df.loc[df['total_folios_crore'] >= t].copy()
    hit = hit[hit['month'] >= pd.Timestamp('2022-01-01')]
    if hit.empty:
        continue
    first = hit.iloc[0]
    milestones.append({
        'month': pd.Timestamp(first['month']),
        'value': float(first['total_folios_crore']),
        'label': f'First ≥ {t:.0f} Cr'
    })

# Required endpoints
milestones.append({'month': m_start, 'value': float(start_val), 'label': 'Jan 2022'})
milestones.append({'month': m_end, 'value': float(end_val), 'label': 'Dec 2025'})

# Clean + dedupe by month
milestones = [m for m in milestones if pd.notna(m['value'])]
by_month = {m['month']: m for m in milestones}
milestones = sorted(by_month.values(), key=lambda x: x['month'])

# --- Build Plotly figure ---
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df['month'],
        y=df['total_folios_crore'],
        mode='lines+markers',
        name='Total folios (Cr)',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=4, color='#1f77b4'),
    )
)

for m in milestones:
    mk_color = '#d62728' if m['label'] in ['Jan 2022', 'Dec 2025'] else '#ff7f0e'
    fig.add_trace(
        go.Scatter(
            x=[m['month']],
            y=[m['value']],
            mode='markers+text',
            name=m['label'],
            text=[m['label']],
            textposition='top center',
            marker=dict(size=12, color=mk_color, line=dict(width=2, color='white'))
        )
    )

fig.update_layout(
    title='Folio count growth (Industry): Jan 2022 → Dec 2025',
    template='plotly_white',
    xaxis_title='Month',
    yaxis_title='Folio count (Cr)',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

fig.update_xaxes(tickformat='%Y-%m', tickangle=-45)

# Explicit annotations for endpoints
if pd.notna(start_val):
    fig.add_annotation(
        x=m_start,
        y=start_val,
        text=f'Jan 2022: {start_val:.2f} Cr',
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-40,
        font=dict(color='#d62728'),
        bgcolor='rgba(255,255,255,0.8)'
    )

if pd.notna(end_val):
    fig.add_annotation(
        x=m_end,
        y=end_val,
        text=f'Dec 2025: {end_val:.2f} Cr',
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-40,
        font=dict(color='#d62728'),
        bgcolor='rgba(255,255,255,0.8)'
    )

fig.show()
